# Teoria: strategie di generazione del testo (decoding)

## Il problema
Ad ogni passo, GPT-2 non restituisce direttamente "la prossima parola": restituisce una **distribuzione di probabilità** su tutto il vocabolario (quale token è più probabile, quale meno). Una **strategia di decoding** è la regola con cui trasformiamo questa distribuzione in UN token scelto, che poi viene riaggiunto alla sequenza per ripetere il procedimento (esattamente il ciclo autoregressivo visto in `05_Transformer_e_LLM_con_addestramento.ipynb`).

Il modello, il tokenizer e il prompt restano sempre gli stessi in questo notebook: quello che cambia, metodo per metodo, è **solo il criterio di scelta del token successivo**.

## Perché non basta "prendere sempre il più probabile"
Scegliere sempre il token più probabile (deterministico) tende a produrre testo che si **ripete** (lo vedi nell'output di Greedy Search qui sotto: la stessa frase torna più volte). Introdurre un po' di casualità (sampling) rende il testo più vario, ma **troppa** casualità lo rende incoerente. Le diverse tecniche sono modi diversi di bilanciare questo compromesso **coerenza vs varietà**.

## Matrice di confronto dei metodi

| Metodo | Idea di base | Deterministico o casuale? | Parametro chiave | Pro | Contro |
|---|---|---|---|---|---|
| **Greedy Search** | Ad ogni passo scegli SEMPRE il token con probabilità più alta. | Deterministico (`do_sample=False`): stesso prompt → sempre stesso output. | nessuno | Semplice, veloce, riproducibile. | Tende a ripetersi (finisce spesso in loop di frasi identiche). |
| **Beam Search** | Mantieni contemporaneamente più sequenze candidate ("beam") invece di una sola, e alla fine scegli la sequenza complessiva più probabile. | Deterministico (`do_sample=False`), ma esplora più ipotesi di Greedy. | `num_beams` (quante sequenze candidate mantenere) | Trova sequenze complessivamente più probabili di Greedy; frasi spesso più sensate. | Più costoso computazionalmente; può comunque ripetersi; tende a preferire risposte "sicure" e generiche. |
| **Top-k Sampling** | Restringi la scelta ai `k` token più probabili, poi scegline uno a caso (campionamento) tra questi `k`, pesato sulla loro probabilità. | Casuale (`do_sample=True`): output diverso ad ogni esecuzione. | `top_k` (quanti token candidati considerare, es. 50) | Più varietà di Greedy/Beam; evita token assurdi perché il pool è limitato a k. | `k` fisso: in contesti dove il modello è molto sicuro, k potrebbe includere token comunque poco sensati; in contesti incerti, k potrebbe escludere token validi. |
| **Nucleus Sampling (Top-p)** | Restringi la scelta al più piccolo gruppo di token la cui probabilità cumulativa raggiunge una soglia `p` (es. 92%), poi campiona tra questi. | Casuale (`do_sample=True`). | `top_p` (soglia di probabilità cumulativa, es. 0.92) | La dimensione del gruppo di candidati si adatta da sola: pochi token se il modello è "sicuro", molti se è "incerto". Spesso più naturale di Top-k. | Testo comunque variabile ad ogni esecuzione; se `p` è troppo alto il pool può diventare grande e includere token poco sensati. |
| **Top-k + Top-p** | Applica prima il filtro Top-k, poi restringe ulteriormente con Top-p, infine campiona tra i token rimasti. | Casuale (`do_sample=True`). | `top_k` **e** `top_p` insieme | Combina i due filtri: elimina sia i token troppo improbabili (top-p) sia le code lunghissime della distribuzione (top-k). | Due parametri da bilanciare invece di uno; più difficile da intuire "a occhio" l'effetto netto. |

## In sintesi
- Se ti serve **riproducibilità** e un output sempre uguale → Greedy o Beam Search (deterministici).
- Se ti serve **varietà/creatività** nel testo generato → Top-k, Nucleus o la combinazione dei due (sampling, casuali).
- Nucleus (Top-p) è generalmente preferito a Top-k puro perché si adatta al livello di "certezza" del modello passo per passo, invece di usare sempre lo stesso numero fisso di candidati.

Nel codice qui sotto, il parametro `technique` della funzione `generate_text` seleziona uno di questi 5 metodi: confronta gli output stampati in fondo per vedere concretamente queste differenze sullo stesso prompt.


In [1]:
# ============================================================
# IMPORTAZIONE DELLE LIBRERIE
# ============================================================

import torch

# Importiamo da Hugging Face Transformers:
# - GPT2LMHeadModel: il modello GPT-2 vero e proprio
# - GPT2Tokenizer: il tokenizer utilizzato da GPT-2
from transformers import GPT2LMHeadModel, GPT2Tokenizer


# ============================================================
# CARICAMENTO DEL MODELLO E DEL TOKENIZER
# ============================================================

# Nome del modello che vogliamo utilizzare.
# "gpt2" è la versione base di GPT-2.
# Esistono anche varianti del modello più grandi, ad esempio:
# "gpt2-medium", "gpt2-large", ecc.
model_name = "gpt2"

# Carichiamo il modello PRE-ADDESTRATO.
#
#
# Qui NON stiamo addestrando GPT-2.
# Stiamo caricando un modello che ha già imparato
# a prevedere i token successivi durante il suo addestramento.
model = GPT2LMHeadModel.from_pretrained(model_name)

# Carichiamo il tokenizer corrispondente al modello.
#
# Il tokenizer trasforma:
#
#     testo → token → numeri (ID)
#
# GPT-2 lavora infatti con rappresentazioni numeriche,
# non direttamente con le parole scritte.
tokenizer = GPT2Tokenizer.from_pretrained(model_name)


# ============================================================
# FUNZIONE PER LA GENERAZIONE DEL TESTO
# ============================================================

# Questa funzione rappresenta il cuore dell'esempio.
#
# Il procedimento generale è SEMPRE lo stesso:
#
#     PROMPT
#       ↓
#     TOKENIZZAZIONE
#       ↓
#     GPT-2
#       ↓
#     probabilità dei possibili token successivi
#       ↓
#     scelta del token
#       ↓
#     nuovo token aggiunto alla sequenza
#       ↓
#     si ripete...
#
# Quello che cambia tra le varie tecniche è soprattutto
# il modo in cui viene scelto il prossimo token.
#
# prompt:
#     testo iniziale da cui partire
#
# technique:
#     strategia utilizzata per scegliere i token
#
# max_length:
#     lunghezza massima della sequenza generata
#
# **kwargs:
#     permette di passare parametri aggiuntivi,
#     come top_k e top_p.
def generate_text(prompt, technique='greedy', max_length=50, **kwargs):

    # --------------------------------------------------------
    # TOKENIZZAZIONE DEL PROMPT
    # --------------------------------------------------------

    # Trasformiamo il testo iniziale in una sequenza di token.
    #
    # return_tensors='pt' significa:
    # restituisci il risultato come Tensor PyTorch.
    #
    # Concettualmente:
    #
    # "The future of Generative AI is"
    #                 ↓
    #              tokenizer
    #                 ↓
    #          [12, 345, 78, ...]
    #
    # Gli ID numerici rappresentano i token del vocabolario
    # di GPT-2.
    inputs = tokenizer.encode(prompt, return_tensors='pt')


    # ========================================================
    # SCELTA DELLA TECNICA DI GENERAZIONE
    # ========================================================

    # ATTENZIONE:
    #
    # Da questo punto in poi il modello GPT-2 è sempre lo stesso.
    #
    # Non cambiamo:
    # - modello
    # - prompt
    # - tokenizer
    #
    # Cambiamo solamente il METODO UTILIZZATO
    # per scegliere il prossimo token.


    # ========================================================
    # 1. GREEDY SEARCH
    # ========================================================

    if technique == 'greedy':

        # Greedy Search è la strategia più semplice:
        #
        # "Prendi sempre il token con la probabilità più alta."
        #
        # Esempio:
        #
        #     token A → 40%
        #     token B → 25%
        #     token C → 15%
        #
        # Greedy sceglie A.
        #
        # Poi il modello calcola nuovamente le probabilità
        # per il token successivo e sceglie ancora quello
        # con probabilità maggiore.
        #
        # do_sample=False:
        # NON facciamo campionamento casuale.
        # La scelta è deterministica.
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                do_sample=False
            )


    # ========================================================
    # 2. BEAM SEARCH
    # ========================================================

    elif technique == 'beam_search':

        # Beam Search cerca di considerare più possibilità
        # contemporaneamente.
        #
        # Invece di dire semplicemente:
        #
        #     "Qual è il token migliore ADESSO?"
        #
        # cerca di mantenere diverse sequenze candidate
        # e confrontarle durante la generazione.
        #
        # num_beams=5 significa che vengono mantenute
        # 5 ipotesi/sequenze candidate.
        #
        # È quindi una ricerca più ampia rispetto a Greedy.
        #
        # Anche qui:
        # do_sample=False
        #
        # quindi non stiamo facendo sampling casuale.
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                num_beams=5,
                early_stopping=True
            )


    # ========================================================
    # 3. TOP-K SAMPLING
    # ========================================================

    elif technique == 'top_k':

        # Top-k cambia approccio:
        #
        # NON scegliamo necessariamente il token più probabile.
        #
        # Prima prendiamo solo i K token più probabili.
        #
        # Ad esempio:
        #
        #     top_k = 50
        #
        # significa:
        #
        # "considera soltanto i 50 token più probabili."
        #
        # Tutti gli altri token vengono esclusi.
        top_k = kwargs.get('top_k', 50)

        # Dopo aver ristretto la scelta ai top-k token,
        # ne viene estratto uno attraverso il SAMPLING.
        #
        # Quindi:
        #
        #     Top-k → restringe i candidati
        #              ↓
        #          sampling
        #              ↓
        #        token scelto
        #
        # do_sample=True è quindi molto importante.
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                top_k=top_k,
                do_sample=True
            )


    # ========================================================
    # 4. NUCLEUS SAMPLING / TOP-P
    # ========================================================

    elif technique == 'nucleus':

        # Top-p funziona in modo diverso da Top-k.
        #
        # Top-k dice:
        #
        #     "Prendi i migliori K token."
        #
        # Top-p invece dice:
        #
        #     "Prendi i token necessari per raggiungere
        #      una probabilità cumulativa pari a p."
        #
        # Esempio:
        #
        #     top_p = 0.92
        #
        # significa che consideriamo il gruppo più piccolo
        # di token la cui probabilità cumulativa raggiunge
        # circa il 92%.
        #
        # Il numero di token considerati quindi NON è fisso.
        top_p = kwargs.get('top_p', 0.92)

        # Anche qui facciamo sampling.
        #
        # Quindi il token più probabile NON è necessariamente
        # quello che verrà scelto.
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                top_p=top_p,
                do_sample=True
            )


    # ========================================================
    # 5. TOP-K + TOP-P
    # ========================================================

    elif technique == 'top_p_and_top_k':

        # Possiamo combinare le due strategie:
        #
        #     Top-k
        #         +
        #     Top-p
        #
        # In questo esempio:
        #
        #     top_k = 50
        #     top_p = 0.92
        #
        # Vengono quindi applicati entrambi i criteri
        # per restringere l'insieme dei token candidati.
        top_k = kwargs.get('top_k', 50)
        top_p = kwargs.get('top_p', 0.92)

        # Alla fine viene effettuato il sampling
        # tra i token rimasti.
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                top_k=top_k,
                top_p=top_p,
                do_sample=True
            )


    # ========================================================
    # TECNICA NON RICONOSCIUTA
    # ========================================================

    else:

        # Se l'utente inserisce una tecnica diversa
        # da quelle previste, generiamo un errore.
        raise ValueError(
            "Tecnica di generazione non valida. "
            "Scegli tra 'greedy', 'beam_search', "
            "'top_k', 'nucleus', 'top_p_and_top_k'."
        )


    # ========================================================
    # DECODIFICA DEL RISULTATO
    # ========================================================

    # model.generate() restituisce una sequenza di ID numerici.
    #
    # Dobbiamo quindi riconvertirla in testo.
    #
    # Facciamo:
    #
    #     ID numerici
    #          ↓
    #        token
    #          ↓
    #        testo
    #
    # decode() esegue questa conversione.
    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Restituiamo il testo generato.
    return generated_text


# ============================================================
# TEST DELLE DIVERSE TECNICHE
# ============================================================

# Questo è il testo iniziale dato a GPT-2.
prompt = "The future of Generative AI is"


# ------------------------------------------------------------
# GREEDY SEARCH
# ------------------------------------------------------------

print("Greedy Search:")

# Usiamo la stessa funzione di prima,
# specificando semplicemente la tecnica.
print(generate_text(
    prompt,
    technique='greedy'
))


# ------------------------------------------------------------
# BEAM SEARCH
# ------------------------------------------------------------

print("\nBeam Search:")

print(generate_text(
    prompt,
    technique='beam_search'
))


# ------------------------------------------------------------
# TOP-K SAMPLING
# ------------------------------------------------------------

print("\nTop-k Sampling:")

# In questo caso diciamo esplicitamente:
#
#     considera i 50 token più probabili
#
# e fai sampling tra questi.
print(generate_text(
    prompt,
    technique='top_k',
    top_k=50
))


# ------------------------------------------------------------
# NUCLEUS / TOP-P SAMPLING
# ------------------------------------------------------------

print("\nNucleus Sampling:")

# Qui utilizziamo top_p = 0.92.
#
# Il numero di token candidati non è fisso:
# dipende dalla distribuzione delle probabilità
# prodotta dal modello.
print(generate_text(
    prompt,
    technique='nucleus',
    top_p=0.92
))


# ------------------------------------------------------------
# TOP-K + TOP-P
# ------------------------------------------------------------

print("\nTop-p and Top-k Sampling:")

# Combiniamo entrambe le strategie:
#
#     top_k = 50
#     top_p = 0.92
#
# e infine effettuiamo il sampling.
print(generate_text(
    prompt,
    technique='top_p_and_top_k',
    top_k=50,
    top_p=0.92
))

c:\Users\066321758\Downloads\28-29_10 - Word embeddings e RNN\07 - Architettura LLM_Corsini\Esercitazioni\venv-transformer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4346.82it/s]


Greedy Search:
The future of Generative AI is in the hands of the people.

The future of Generative AI is in the hands of the people.

The future of Generative AI is in the hands of the people.

The future

Beam Search:
The future of Generative AI is in the hands of the next generation of AI researchers.

The future of Generative AI is in the hands of the next generation of AI researchers.

The future of Generative AI is in the hands

Top-k Sampling:
The future of Generative AI is going to look a lot better without it. This one was brought to us by The Brain Scientist. If you have not heard of The Brain Scientist, his TED talk on Artificial Intelligence started as a little known little project

Nucleus Sampling:
The future of Generative AI is unknown, but there is no denying the power of AI to disrupt society. It is possible that the technology that has been developed by Artificial Intelligence could potentially transform our lives.

The Future of AI

Gener

Top-p and Top-k Sampling:
T